In [1]:
# API 키를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

from langchain.chains import RetrievalQA
from langchain.llms import OpenAI
import os

import nltk

from langchain.document_loaders import DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import SentenceTransformerEmbeddings
from langchain.vectorstores import Chroma


# API 키 정보 로드
load_dotenv()


True

In [2]:
def add_to_chroma_batched(texts, embeddings, batch_size=32):
    db = None  # 초기화
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        if db is None:
            db = Chroma.from_documents(batch_texts, embeddings)
        else:
          db.add_documents(batch_texts)

    return db

In [ ]:
nltk.download('averaged_perceptron_tagger_eng')

# 텍스트 파일이 있는 폴더 경로
folder_path = "./data/crawled_data"

# 텍스트 파일 로드
loader = DirectoryLoader(folder_path, glob="./*.txt")
documents = loader.load()

# 텍스트 분할
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
texts = text_splitter.split_documents(documents)[:]

# 임베딩 모델 및 벡터 데이터베이스 설정
embeddings = SentenceTransformerEmbeddings(model_name="paraphrase-multilingual-mpnet-base-v2")

chroma_db = add_to_chroma_batched(texts, embeddings, batch_size=64)

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\SSAFY\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_33868\3885264151.py:15: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = SentenceTransformerEmbeddings(model_name="paraphrase-multilingual-mpnet-base-v2")
c:\Users\SSAFY\anaconda3\envs\rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook imp

In [ ]:
retriever = chroma_db.as_retriever(search_kwargs={'k': 3})

# 언어 모델 및 RAG 체인 설정
llm = OpenAI(max_tokens=256)
qa = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=retriever)

# 질문 입력 및 답변 생성
query = """
당신은 아이들의 연령, 관심사, 상황에 맞춰 활동을 추천해주는 전문가입니다.
사용자는 당신이 추천한 활동을 아이들에게 시키고, 수행을 완료했다면 보상을 지금할 것입니다.

[입력 정보]:
- 아이의 연령: {age}
- 아이의 관심사: {interests} (예: 그림 그리기, 과학 실험, 운동 등 / 없으면 무시시)

[출력 형식]:
1. 활동 제목: ...
2. 활동 설명: ...
3. 필요한 준비물 (선택 사항): ...
4. 예상 소요 시간 (선택 사항): ...
5. 추가 팁 (선택 사항): ..."""
result = qa.run(query)

print(result)


1. 활동 제목: 집안일 놀이로 하는 집콕육아
2. 활동 설명: 아이에게 집안일을 가르치는 것은 아이의 자립심과 책임감을 키우는 데 도움이 됩니다. 집안일 놀이를 통해 아이는 즐겁게 공부하고 실생활에서 필요한 기술을 익힐 수 있습니다.
3. 필요한 준비물: 일반적으로 집안일을 할 때 사용하는 물건들 (수저, 휴지, 물 등)
4. 예상 소요 시간: 30분 ~ 1시간
5. 추가 팁:
- 아이에게 미션 놀이처럼 심부름을 시키면 더 재미있게 할 수 있습니다.
- 심부름 성공
